# Bronze layer


In [0]:
table_name = 'flightdata.bronze.flightData'
source_data = '/Volumes/flightdata/landing-data/landing-vol/flights.csv'
source_format = 'CSV'

# Drop the existing (incorrectly-schema'd) table so it can be recreated properly
spark.sql("DROP TABLE IF EXISTS " + table_name)

spark.sql("CREATE TABLE IF NOT EXISTS " + table_name)

spark.sql("COPY INTO " + table_name + \
  " FROM '" + source_data + "'" + \
  " FILEFORMAT = " + source_format + \
  " FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true', 'multiLine' = 'true')" + \
  " COPY_OPTIONS ('mergeSchema' = 'true')"
)

# Verify
display(spark.sql("SELECT * FROM " + table_name + " LIMIT 5"))
spark.sql("DESCRIBE " + table_name).show(50, truncate=False)


In [0]:
%sql
select * from flightdata.bronze.flightdata


# migrate to silver schema

In [0]:
df.show(2, truncate=False)

In [0]:
from pyspark.sql import functions as F

from pyspark.sql import functions as F

# Read the corrected Bronze table
df = spark.read.table("flightdata.bronze.flightData")

# Sanity check — confirm types look right before merging
df.printSchema()

# Merge year/month/day into a single date column
df_silver = df.withColumn(
    "date",
    F.to_date(
        F.concat_ws("-", F.col("year"), F.col("month"), F.col("day")),
        "yyyy-M-d"
    )
).drop("year", "month", "day")

# Write to Silver as a managed Delta table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("flightdata.silver.flightdata")

# Verify
display(spark.read.table("flightdata.silver.flightdata"))

In [0]:
%sql
select * from flightdata.silver.flightdata

## check for duplicates.

In [0]:
df_silver = spark.read.table("flightdata.silver.flightdata")

total_count = df_silver.count()
distinct_count = df_silver.distinct().count()

print(f"Total rows: {total_count}")
print(f"Distinct rows: {distinct_count}")
print(f"Duplicate rows: {total_count - distinct_count}")

from pyspark.sql import functions as F

dup_rows = df_silver.groupBy(df_silver.columns).count().filter(F.col("count") > 1)
display(dup_rows)

total_count = df_silver.count()
distinct_count = df_silver.distinct().count()
print(f"Total rows: {total_count}, Distinct rows: {distinct_count}")

## Check for nulls

In [0]:
from pyspark.sql import functions as F

df_silver = spark.read.table("flightdata.silver.flightdata")

# Count nulls per column
null_counts = df_silver.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_silver.columns]
)

display(null_counts)

# Gold-Business logic aggreagtes

## Delay analysis — avg delay by carrier and by origin airport

In [0]:
from pyspark.sql import functions as F

df_silver = spark.read.table("flightdata.silver.flightdata")

# Avg delay by carrier
delay_by_carrier = df_silver.groupBy("carrier").agg(
    F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
    F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
    F.count("*").alias("flight_count")
).orderBy(F.desc("avg_dep_delay"))

# Avg delay by origin airport
delay_by_origin = df_silver.groupBy("origin").agg(
    F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
    F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
    F.count("*").alias("flight_count")
).orderBy(F.desc("avg_dep_delay"))

delay_by_carrier.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.delay_by_carrier")

delay_by_origin.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.delay_by_origin")

## Flight volume trends — by route, by date, by carrier

In [0]:
# Volume by route (origin -> dest)
volume_by_route = df_silver.groupBy("origin", "dest").agg(
    F.count("*").alias("flight_count")
).orderBy(F.desc("flight_count"))

# Volume by date
volume_by_date = df_silver.groupBy("date").agg(
    F.count("*").alias("flight_count")
).orderBy("date")

# Volume by carrier
volume_by_carrier = df_silver.groupBy("carrier").agg(
    F.count("*").alias("flight_count")
).orderBy(F.desc("flight_count"))

volume_by_route.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.volume_by_route")

volume_by_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.volume_by_date")

volume_by_carrier.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.volume_by_carrier")

## Airport/carrier performance summary — combined view

In [0]:
performance_summary = df_silver.groupBy("carrier", "origin").agg(
    F.count("*").alias("total_flights"),
    F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
    F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
    F.round(F.avg("air_time"), 2).alias("avg_air_time"),
    F.round((F.sum(F.when(F.col("arr_delay") <= 0, 1).otherwise(0)) / F.count("*")) * 100, 2)
        .alias("on_time_pct")
).orderBy(F.desc("total_flights"))

performance_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.performance_summary")

In [0]:
display(spark.read.table("flightdata.gold.delay_by_carrier"))
display(spark.read.table("flightdata.gold.volume_by_date"))
display(spark.read.table("flightdata.gold.performance_summary"))